In [ ]:
import pandas as pd
import json
import re
import requests
from bs4 import BeautifulSoup
from openai import OpenAI
from google.colab import userdata

### Processing the official arxiv taxonomy descriptions

In [ ]:
#acm codes used in the arxiv taxonomy
def parse_acm_mapping(filepath):
    acm_map = {}

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            #split on first colon
            parts = line.split(":", 1)
            if len(parts) == 2:
                code = parts[0].strip()
                name = parts[1].strip()
                acm_map[code] = name

    return acm_map

acm_map = parse_acm_mapping("acm.txt")

In [ ]:
acm_map

{'A.': 'General Literature',
 'A.0': 'GENERAL',
 'A.1': 'INTRODUCTORY AND SURVEY',
 'A.2': 'REFERENCE (e.g., dictionaries, encyclopedias, glossaries)',
 'A.m': 'MISCELLANEOUS',
 'B.': 'Hardware',
 'B.0': 'GENERAL',
 'B.1': 'CONTROL STRUCTURES AND MICROPROGRAMMING',
 'B.1.0': 'General',
 'B.1.1': 'Control Design Styles',
 'B.1.2': 'Control Structure Performance Analysis and Design Aids',
 'B.1.3': 'Control Structure Reliability, Testing, and Fault-Tolerance',
 'B.1.4': 'Microprogram Design Aids',
 'B.1.5': 'Microcode Applications',
 'B.1.m': 'Miscellaneous',
 'B.2': 'ARITHMETIC AND LOGIC STRUCTURES',
 'B.2.0': 'General',
 'B.2.1': 'Design Styles',
 'B.2.2': 'Performance Analysis and Design Aids',
 'B.2.3': 'Reliability, Testing, and Fault-Tolerance',
 'B.2.4': 'High-Speed Arithmetic',
 'B.2.m': 'Miscellaneous',
 'B.3': 'MEMORY STRUCTURES',
 'B.3.0': 'General',
 'B.3.1': 'Semiconductor Memories',
 'B.3.2': 'Design Styles',
 'B.3.3': 'Performance Analysis and Design Aids',
 'B.3.4': 'Reli

In [ ]:
def enrich_description(description, acm_map):
    # build one that matches any acm code, longest first
    sorted_codes = sorted(acm_map.keys(), key=len, reverse=True)
    pattern = '|'.join(re.escape(code) for code in sorted_codes)

    def replace_match(match):
        code = match.group(0)
        return f"{code} ({acm_map[code]})"

    return re.sub(pattern, replace_match, description)

In [ ]:
df = pd.read_excel("arxiv_label_descriptions.xlsx")

df["enriched_description"] = df["description"].fillna("").apply(lambda desc: enrich_description(desc, acm_map))

label_descriptions = {
    row["label"]: {
        "label_expanded": row["label_expanded"],
        "description": row["enriched_description"] }
    for _, row in df.iterrows()}

In [ ]:
label_descriptions
#looks ok
#two label descriptions are missing, because they were not in the taxonomy

{'Adaptation and Self-Organizing Systems': {'label_expanded': 'Adaptation and Self-Organizing Systems (Physics)',
  'description': 'Adaptation, self-organizing systems, statistical physics, fluctuating systems, stochastic processes, interacting particle systems, machine learning'},
 'Applications': {'label_expanded': 'Applications (Statistics)',
  'description': 'Biology, Education, Epidemiology, Engineering, Environmental Sciences, Medical, Physical Sciences, Quality Control, Social Sciences'},
 'Artificial Intelligence': {'label_expanded': 'Artificial Intelligence (Computer Science)',
  'description': 'Covers all areas of AI except Vision, Robotics, Machine Learning, Multiagent Systems, and Computation and Language (Natural Language Processing), which have separate subject areas. In particular, includes Expert Systems, Theorem Proving (although this may overlap with Logic in Computer Science), Knowledge Representation, Planning, and Uncertainty in AI. (Computing Methodologies) Roughl

In [ ]:
with open("arxiv_label_descriptions_expanded.json", "w", encoding="utf-8") as f:
    json.dump(label_descriptions, f, indent=2, ensure_ascii=False)

### Generating more suitable descriptions (based on the official ones) with gpt-4o-mini

In [ ]:
api_key = userdata.get("open")
client = OpenAI(api_key=api_key)

In [ ]:
input_path = "arxiv_label_descriptions_expanded.json"

with open(input_path, "r", encoding="utf-8") as f:
    official_descriptions = json.load(f)


items = [
    {
        "label": label,
        "label_expanded": info["label_expanded"],
        "description": info["description"]
    }
    for label, info in official_descriptions.items()]

all_labels = sorted(official_descriptions.keys())
all_labels_text = "\n".join(f"- {label}" for label in all_labels)

In [ ]:
#skipping the label_expanded field as I want it to be used only by the LLM to get a better undestanding of the orignal labels, there's no need for it in the output

schema = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "items": {
            "type": "array",
            "minItems": len(items),
            "maxItems": len(items),
            "items": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "label": {
                        "type": "string",
                        "description": "The exact label from the input."},
                    "description": {
                        "type": "string",
                        "description": "A concise classification-oriented label description."}},
                "required": ["label", "description"]}}},
    "required": ["items"]}

In [ ]:
system_prompt = """
You are an expert in computer science, arXiv subject categories, and multi-label academic text classification.

Your task is to rewrite official arXiv category descriptions into concise, classification-oriented label descriptions.
The rewritten descriptions will be used in a prompt for another LLM that must classify academic abstracts into fine-grained AAPD labels.

The classifier will struggle most with categories that sound similar.
Your descriptions must make the distinctions crystal clear.
When the official description is missing you may use your domain knowledge of computer science and academic publishing to produce a useful description.
Return only the structured JSON output.
"""

user_prompt = f"""
Rewrite the following label descriptions to make them more useful for multi-label classification.

The final classifier must choose among these 54 labels:
{all_labels_text}

Requirements for each rewritten description:
- Use 2-3 concise sentences.
- Sentence 1: What specific kinds of papers and topics belong here.
- Sentence 2: What this category explicitly does NOT cover.
- Sentence 3 (only if needed): Any additional distinguishing detail.
- Use the label_expanded field to understand the domain context (e.g. Statistics, Physics, Mathematics)..
- Do not add topics that are not supported by the official description.
- Do not use vague phrases like "distinct from other fields" — always name the specific similar label.
- Keep the original label string exactly unchanged.
- Return exactly one item for every input item.

Input label descriptions:
{json.dumps(items, indent=2, ensure_ascii=False)}
"""

response = client.responses.create(
    model="gpt-4o-mini",
    temperature=0,
    input=[
        {
            "role": "system",
            "content": system_prompt},
        {
            "role": "user",
            "content": user_prompt}],
    text={
        "format": {
            "type": "json_schema",
            "name": "aapd_prompt_friendly_descriptions",
            "schema": schema,
            "strict": True}})

result = json.loads(response.output_text)
rewritten_df = pd.DataFrame(result["items"])

In [ ]:
#checking whether the labels are correct
expected = set(official_descriptions.keys())
returned = set(rewritten_df["label"])
if expected != returned:
    raise ValueError(f"Label mismatch. Missing: {expected - returned}, Extra: {returned - expected}")

In [ ]:
rewritten_descriptions = dict(zip(rewritten_df["label"], rewritten_df["description"]))
with open("aapd_label_descriptions_prompt_friendly.json", "w", encoding="utf-8") as f:
    json.dump(rewritten_descriptions, f, indent=2, ensure_ascii=False)
